# Build ABO Physics Subset

Этот ноутбук пересобирает подмножество Amazon ABO для задачи извлечения физических свойств.

Выход:
- `dataset/abo_physics_val/meta.json`
- `dataset/abo_physics_val/images/...`
- `dataset/abo_physics_val/summary.json`


In [1]:
# !pip install -q boto3 rembg onnxruntime


In [2]:
import subprocess

# Настройки генерации
LISTING_SHARDS = "0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15"
MAX_SAMPLES = 240
MIN_KNOWN_PROPERTIES = 4
MAX_PER_PRODUCT_TYPE = 25
EXCLUDE_PRODUCT_TYPES = "CELLULAR_PHONE_CASE,PORTABLE_ELECTRONIC_DEVICE_COVER"
SEED = 42
DOWNLOAD_MISSING = True  # Для Colab обычно True
GENERATE_MASKS = True
MASK_BACKEND = "rembg"  # rembg | simple

cmd = [
    "python", "scripts/build_abo_physics_subset.py",
    "--dataset-root", "dataset",
    "--cache-dir", "dataset/abo_vlm_val/_cache",
    "--source-images-dir", "dataset/abo_vlm_val/images",
    "--listing-shards", LISTING_SHARDS,
    "--max-samples", str(MAX_SAMPLES),
    "--min-known-properties", str(MIN_KNOWN_PROPERTIES),
    "--max-per-product-type", str(MAX_PER_PRODUCT_TYPE),
    "--exclude-product-types", EXCLUDE_PRODUCT_TYPES,
    "--seed", str(SEED),
    "--clear-output",
]

if DOWNLOAD_MISSING:
    cmd.append("--download-missing")
if GENERATE_MASKS:
    cmd += [
        "--generate-masks",
        "--mask-backend", MASK_BACKEND,
        "--min-mask-area-ratio", "0.01",
        "--max-mask-area-ratio", "0.95",
    ]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: python scripts/build_abo_physics_subset.py --dataset-root dataset --cache-dir dataset/abo_vlm_val/_cache --source-images-dir dataset/abo_vlm_val/images --listing-shards 0,1 --max-samples 240 --min-known-properties 4 --max-per-product-type 25 --seed 42 --clear-output --download-missing --generate-masks --mask-backend simple --min-mask-area-ratio 0.01 --max-mask-area-ratio 0.95
Wrote: dataset/abo_physics_val/meta.json
Wrote: dataset/abo_physics_val/summary.json
Samples: 217


CompletedProcess(args=['python', 'scripts/build_abo_physics_subset.py', '--dataset-root', 'dataset', '--cache-dir', 'dataset/abo_vlm_val/_cache', '--source-images-dir', 'dataset/abo_vlm_val/images', '--listing-shards', '0,1', '--max-samples', '240', '--min-known-properties', '4', '--max-per-product-type', '25', '--seed', '42', '--clear-output', '--download-missing', '--generate-masks', '--mask-backend', 'simple', '--min-mask-area-ratio', '0.01', '--max-mask-area-ratio', '0.95'], returncode=0)

In [3]:
import json
from pathlib import Path

meta_path = Path("dataset/abo_physics_val/meta.json")
summary_path = Path("dataset/abo_physics_val/summary.json")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("Samples:", len(meta))
print("\nProperty distributions:")
for k, v in summary.get("property_distributions", {}).items():
    print(f"- {k}: {v}")

if meta:
    print("\nFirst sample:")
    print(json.dumps(meta[0], ensure_ascii=False, indent=2)[:1200])


Samples: 217

Property distributions:
- material: {'mixed': 4, 'metal': 24, 'wood': 20, 'plastic': 20, 'glass': 23, 'stone': 19, 'paper': 22, 'fabric': 50, 'rubber': 24, 'ceramic': 11}
- rigidity: {'mixed': 4, 'rigid': 117, 'flexible': 46, 'soft': 50}
- transparency: {'translucent': 1, 'opaque': 189, 'transparent': 25, 'unknown': 2}
- surface: {'mixed': 46, 'smooth': 97, 'rough': 53, 'fuzzy': 18, 'porous': 3}
- fragility: {'unknown': 2, 'durable': 161, 'fragile': 54}

First sample:
{
  "image_id": "21SgHQ5qBvL",
  "path": "abo_physics_val/images/77/77398330.jpg",
  "primary_object": "phone accessory",
  "properties": {
    "material": "mixed",
    "rigidity": "mixed",
    "transparency": "translucent",
    "surface": "mixed",
    "fragility": "unknown"
  },
  "notes": "metadata-derived physical pseudo-label",
  "abo_meta": {
    "item_id": "B0093VGUFG",
    "product_type": "PHONE_ACCESSORY",
    "domain_name": "amazon.co.uk",
    "title": "AmazonBasics Translucent Protective TPU Plasti